# 8장 실습: Text2Cypher (POLE 데이터셋)

실습 가이드 8장 8-1 ~ 8-4의 코드를 순서대로 실행하는 노트북입니다. 설명은 실습 가이드를 함께 봅니다.

| 항목 | 내용 |
|---|---|
| 연결할 DB | POLE (샌드박스 또는 Docker) |
| .env 설정 | POLE DB의 NEO4J 네 줄 |

.env를 바꿨다면 노트북 상단의 `Restart`로 커널을 재시작한 뒤 처음부터 실행합니다. 연결 셀의 `connect_graph()`는 .env를 다시 읽지만, 앞 셀에서 만든 객체가 이전 DB를 가리킬 수 있으므로 처음부터 다시 실행합니다.

## 연결 확인

`Neo4jGraph`는 연결할 때 DB 스키마를 읽습니다. 이때 APOC 플러그인을 쓰므로, APOC가 없는 DB에서는 `Could not use APOC procedures` 오류가 납니다(Sandbox, AuraDB, 실습용 Docker 이미지에는 들어 있습니다).

출력된 레이블별 노드 수로 이 차시에 맞는 DB에 연결됐는지 확인합니다.

In [ ]:
from tools.graphrag_tools import connect_graph

# .env의 접속 정보로 연결하고 레이블별 노드 수를 출력한다. expect의 레이블이 없으면 다른 DB에 연결됐다고 알려 준다.
# NEO4J는 접속 정보 묶음이다. 다른 LangChain 객체를 만들 때 **NEO4J로 그대로 넘긴다(실습 가이드 6-2).
graph, NEO4J = connect_graph(expect=["Crime", "Location", "PostCode"])

## 1. 스키마 확인 (실습 가이드 8-1)

Text2Cypher 프롬프트에 넣을 스키마입니다. 데이터 구조가 바뀌었으면 `refresh_schema()`로 다시 읽습니다.

In [ ]:
graph.refresh_schema()
print(graph.schema)

## 2. OpenAI API로 직접 구현 (실습 가이드 8-3)

few-shot 예시는 user와 assistant 메시지 쌍으로 넣습니다. LLM을 두 번 호출합니다. 첫 번째는 쿼리를 만들고, 두 번째는 쿼리 결과를 문장으로 바꿉니다.

In [ ]:
from openai import OpenAI

client = OpenAI()

SYSTEM = """너는 Neo4j Cypher 전문가다. 아래 스키마에 있는 레이블, 관계, 속성만 사용해
사용자 질문에 답하는 Cypher 쿼리 하나를 작성한다. 설명 없이 쿼리만 출력한다.

스키마:
{schema}"""

FEW_SHOTS = [
    ("Piccadilly에서 발생한 범죄 건수는?",
     "MATCH (c:Crime)-[:OCCURRED_AT]->(l:Location {address: 'Piccadilly'})\nRETURN count(c) AS crime_count"),
    ("M22 8ER 우편번호 지역의 범죄 유형 분포는?",
     "MATCH (pc:PostCode {code: 'M22 8ER'})<-[:HAS_POSTCODE]-(l:Location)<-[:OCCURRED_AT]-(c:Crime)\nRETURN c.type, count(*) AS count ORDER BY count DESC"),
]

def text_to_cypher(question: str) -> str:
    messages = [{"role": "system", "content": SYSTEM.format(schema=graph.schema)}]
    for q, cypher in FEW_SHOTS:
        messages += [{"role": "user", "content": q}, {"role": "assistant", "content": cypher}]
    messages.append({"role": "user", "content": question})

    resp = client.chat.completions.create(model="gpt-4o", temperature=0, messages=messages)
    cypher = resp.choices[0].message.content.strip()
    return cypher.removeprefix("```cypher").removeprefix("```").removesuffix("```").strip()

def answer(question: str) -> str:
    cypher = text_to_cypher(question)
    rows = graph.query(cypher)
    print("생성된 Cypher:\n", cypher)
    resp = client.chat.completions.create(
        model="gpt-4o", temperature=0,
        messages=[
            {"role": "system", "content": "쿼리 결과만 근거로 한국어로 답한다."},
            {"role": "user", "content": f"질문: {question}\n쿼리 결과: {rows[:20]}"},
        ],
    )
    return resp.choices[0].message.content

print(answer("현재 수사 중인 범죄 사건의 개수는?"))

## 3. GraphCypherQAChain으로 바꾸기 (실습 가이드 8-4)

프롬프트 템플릿 안에서 Cypher의 속성 중괄호는 `{{ }}`로 두 번 씁니다. 한 번만 쓰면 템플릿 변수로 해석돼 오류가 납니다.

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_neo4j import GraphCypherQAChain

CYPHER_TEMPLATE = """너는 Neo4j Cypher 전문가다. 스키마에 있는 레이블, 관계, 속성만 사용해 쿼리 하나를 작성한다.
설명 없이 쿼리만 출력한다.

스키마:
{schema}

예시:
질문: Piccadilly에서 발생한 범죄 건수는?
쿼리: MATCH (c:Crime)-[:OCCURRED_AT]->(l:Location {{address: 'Piccadilly'}}) RETURN count(c) AS crime_count

질문: M22 8ER 우편번호 지역의 범죄 유형 분포는?
쿼리: MATCH (pc:PostCode {{code: 'M22 8ER'}})<-[:HAS_POSTCODE]-(l:Location)<-[:OCCURRED_AT]-(c:Crime) RETURN c.type, count(*) AS count ORDER BY count DESC

질문: {question}
쿼리:"""

cypher_prompt = PromptTemplate(input_variables=["schema", "question"], template=CYPHER_TEMPLATE)

cypher_chain = GraphCypherQAChain.from_llm(
    llm=ChatOpenAI(model="gpt-4o", temperature=0),
    graph=graph,
    cypher_prompt=cypher_prompt,
    validate_cypher=True,             # 관계 방향 오류를 스키마 기준으로 교정
    return_intermediate_steps=True,   # 생성된 쿼리와 조회 결과를 함께 반환
    allow_dangerous_requests=True,    # LLM이 만든 쿼리를 실행하므로 명시적 동의가 필요
    verbose=True,
)

result = cypher_chain.invoke({"query": "범죄자는 아니지만 범죄자를 가장 많이 알고 있는 사람은?"})
print(result["result"])
print(result["intermediate_steps"][0]["query"])

## 실습 과제

위의 셀을 모두 실행한 뒤 진행합니다. 과제마다 문제 아래의 빈 셀에 코드를 작성합니다.

### 과제 1. M22 8ER 지역의 최다 범죄 유형

"M22 8ER 지역에서 가장 많이 발생한 범죄 유형은?"을 2번(직접 구현)과 3번(체인) 방식으로 질문하고, 생성된 쿼리를 정답 쿼리와 비교해 검증하세요.

In [ ]:
# 과제 1: 여기에 작성합니다


### 과제 2. few-shot 예시를 뺐을 때

3번 셀의 `CYPHER_TEMPLATE`에서 예시를 모두 뺀 프롬프트로 체인을 만들고, 과제 1과 같은 질문을 실행해 생성된 쿼리와 결과를 비교하세요.

In [ ]:
# 과제 2: 여기에 작성합니다


### 과제 3. 직접 구현과 체인 비교

아래 질문 3개를 2번의 `text_to_cypher()`와 3번의 `cypher_chain`에 넣어 생성된 쿼리를 나란히 비교하세요.

1. 현재 수사 중인 범죄 사건의 개수는?
2. Piccadilly에서 발생한 범죄 건수는?
3. 범죄자는 아니지만 범죄자를 가장 많이 알고 있는 사람은?

In [ ]:
# 과제 3: 여기에 작성합니다
